## Business Problem

Telecommunication companies face significant revenue loss due to customer churn. Retaining existing customers is more cost-effective than acquiring new ones, making churn prediction a critical business priority.

## Objective

The goal of this project is to develop a machine learning model that predicts customers likely to churn and identifies the key factors driving churn behavior.

## Success Metric

The model prioritizes recall to ensure that the majority of at-risk customers are identified for proactive retention strategies.

In [81]:
%cd /content/drive/MyDrive/ML_Portfolio/Customer-churn-ML

/content/drive/MyDrive/ML_Portfolio/Customer-churn-ML


In [82]:
# ----- Data Handling ------
import pandas as pd
import numpy as np

# ----- Visualization ------
import matplotlib.pyplot as plt
import seaborn as sns


In [83]:
# import dataset
# Verify the contents of the 'data' directory
!ls '/content/drive/MyDrive/ML_Portfolio/Customer-churn-ML/data/'
df = pd.read_csv('/content/drive/MyDrive/ML_Portfolio/Customer-churn-ML/data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

archive.zip			      WA_Fn-UseC_-Telco-Customer-Churn.xls
WA_Fn-UseC_-Telco-Customer-Churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## **Initial Observation:**
###- Dataset contains customer-level telecom data
### - Columns include demographic, service usage, and billing information
### - Target variable 'Churn' indicates whether a customer left

In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Dataset Summary (from `df.info()`):

- **Total Entries:** 7043
- **Total Columns:** 21
- **Data Types:**
  - `object`: 18 columns
  - `int64`: 2 columns
  - `float64`: 1 column
- `TotalCharges` is of type `object` and should likely be numeric, which might indicate hidden missing values or parsing issues. This will need further investigation and cleaning

In [85]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


**Missing Values:** No explicitly missing values shown by `Non-Null Count` for any column

In [86]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


## **Observations:**
 - Dataset is imbalanced (more non-churn than churn)
- This may affect model performance and evaluation

# **Data Cleaning**

In [87]:
# Convert to lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Create a copy of original dataset before cleaning
df_copy = df.copy()

In [88]:
# Remove leading/trailing spaces from categorical columns
for col in df_copy.select_dtypes(include='object').columns:
    df_copy[col] = df_copy[col].str.strip()

In [89]:
# Check duplicates
df_copy.duplicated().sum()

np.int64(0)

In [90]:
# check for inconsistent values
for col in df_copy.select_dtypes(include='object').columns:
    print(col, df_copy[col].unique())
    print('\n')

customerid ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']


gender ['Female' 'Male']


partner ['Yes' 'No']


dependents ['No' 'Yes']


phoneservice ['No' 'Yes']


multiplelines ['No phone service' 'No' 'Yes']


internetservice ['DSL' 'Fiber optic' 'No']


onlinesecurity ['No' 'Yes' 'No internet service']


onlinebackup ['Yes' 'No' 'No internet service']


deviceprotection ['No' 'Yes' 'No internet service']


techsupport ['No' 'Yes' 'No internet service']


streamingtv ['No' 'Yes' 'No internet service']


streamingmovies ['No' 'Yes' 'No internet service']


contract ['Month-to-month' 'One year' 'Two year']


paperlessbilling ['Yes' 'No']


paymentmethod ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


totalcharges ['29.85' '1889.5' '108.15' ... '346.45' '306.6' '6844.5']


churn ['No' 'Yes']




In [91]:
df_copy[df_copy['monthlycharges'] == 0]

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn


In [92]:
# Fix logical inconsistency:
# Customers with tenure = 0 should have totalcharges = 0
df_copy.loc[df_copy['tenure'] == 0, 'totalcharges'] = 0

In [93]:
df_copy[(df_copy['internetservice'] == 'No') & (df_copy['onlinesecurity'] == 'Yes')]

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn


In [94]:
df_copy.describe()

,seniorcitizen,tenure,monthlycharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


# **Data Cleaning Summary:**
 - Standardized column names
 - Created a working copy of dataset
 - Converted totalcharges to numeric and handled missing values
 - Checked and removed duplicates (if any)
 - Ensured consistency in categorical values
 - Validated numerical columns and logical consistency
 - Prepared dataset for exploratory analysis

# **EDA**

In [103]:
!git config --global user.name "davidmba819"
!git config --global user.email "davidmba816@gmail.com"
!git config --global credential.helper store
!git add .
!git commit -m "clean notebook"
!git push https://davidmba819@github.com/davidmba819/Customer-churn-ML.git

[main dda1ce0] clean notebook
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (8/8), 8.14 KiB | 438.00 KiB/s, done.
Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 1 local object.
remote: error: GH013: Repository rule violations found for refs/heads/main.
remote: 
remote: - GITHUB PUSH PROTECTION
remote:   —————————————————————————————————————————
remote:     Resolve the following violations before pushing again
remote: 
remote:     - Push cannot contain secrets
remote: 
remote:     
remote:      (?) Learn how to resolve a blocked push
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push
remote:     
remote:     
remote:      